# FloodSentinel - Sri Lanka Flood Early Warning & Risk Prediction System
## Machine Learning Pipeline (Module Assignment)

**Module:** Machine Learning Group Assignment  
**Team Member (Lead for Steps 1 & 2):** Aloka Fernando  
**Project Team:** FloodSentinel (Group of 4)  
**Domain:** Hydro-Meteorological Disaster Risk Prediction for Sri Lanka  
**Primary Dataset:** `sri_lanka_flood_risk_dataset_25000.csv` (25,000 Records, 32 Features)  
**Target Variable:** `flood_occurrence_current_event` (Binary: Yes / No)  

---

## 📍 Step 1: Problem Definition
**Author / Contributor:** Aloka Fernando  


### 1.1 Real-World Context & Disaster Landscape in Sri Lanka
Sri Lanka is an island nation heavily influenced by two major monsoon seasons:
1. **South-West Monsoon (May to September):** Brings intense precipitation to the Western, Southern, and Sabaragamuwa provinces, frequently inundating the Kelani, Kalu, Gin, and Nilwala river basins.
2. **North-East Monsoon (November to February):** Brings heavy convective rain to the Northern, Eastern, and North-Central dry zones.

Rapid urbanization, loss of natural wetlands, and inadequate drainage infrastructure have escalated flood vulnerability in dense metropolitan areas like Colombo and Gampaha, as well as downstream rural settlements. Floods inflict severe infrastructure damage, agricultural loss, and human displacement annually.

### 1.2 Objective & Machine Learning Formulation
The objective of **FloodSentinel** is to develop a robust, high-recall machine learning classification system capable of predicting active flood occurrence for any given location in Sri Lanka based on:
- **Topographic parameters** (Elevation, distance to nearest major river)
- **Meteorological indices** (7-day cumulative rainfall, monthly rainfall, natural drainage capacity)
- **Satellite Earth Observation indices** (NDVI for vegetation density, NDWI for surface water saturation)
- **Regional infrastructure & demographic metrics** (Built-up percentage, population density, road quality)

### 1.3 Mathematical Formulation & Cost-Sensitive Alert Criteria
We formulate this as a **supervised binary classification problem**:
$$\hat{y} = f(X) \in \{0, 1\}$$
where:
- $y = 0$: No flood occurrence (Normal conditions)
- $y = 1$: Active flood event (Emergency alert triggered)

#### The Asymmetric Disaster Cost Matrix:
In disaster early warning, classification errors have profoundly asymmetric consequences:
- **False Positive ($FP$):** Warning issued, but no flood occurs. Cost: Temporary evacuation inconvenience, economic friction ($C_{FP} = 1$).
- **False Negative ($FN$):** Flood occurs, but no warning is issued. Cost: Loss of human lives, destruction of property, failure of disaster response ($C_{FN} \approx 10 \times C_{FP}$).

Therefore, standard raw classification accuracy is an inappropriate metric. The primary optimization criteria are:
1. **Precision-Recall AUC (PR-AUC):** Optimal evaluation under heavy class imbalance (~9.9% positive class).
2. **Recall / Sensitivity ($> 90\%$):** Ensuring almost all genuine flood events trigger an alert.
3. **Cost-Sensitive Risk Score:** Calibrating predicted probabilities into four actionable operational tiers: `Safe`, `Advisory`, `Warning`, and `Critical Emergency`.

## Step 2: Data Collection & Ingestion
**Author / Contributor:** Aloka Fernando  


### 2.1 Dataset Provenance & Metadata
- **File:** `../sri_lanka_flood_risk_dataset_25000.csv` (or root `sri_lanka_flood_risk_dataset_25000.csv`)
- **Records:** 25,000 spatial observations
- **Attributes:** 32 features encompassing geographic coordinates, elevation, rainfall measurements, satellite indices, and infrastructure attributes across all 25 Sri Lankan administrative districts.

Let us import the necessary analytical libraries and load the raw dataset.

In [ ]:
# Import core scientific and analytical libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visualization styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('Libraries successfully loaded!')

In [ ]:
# Locate and load the Sri Lanka Flood Risk dataset
data_paths = [
    'sri_lanka_flood_risk_dataset_25000.csv',
    '../sri_lanka_flood_risk_dataset_25000.csv',
    'data/raw/sri_lanka_flood_risk_dataset_25000.csv'
]

dataset_path = None
for path in data_paths:
    if os.path.exists(path):
        dataset_path = path
        break

if dataset_path is None:
    raise FileNotFoundError('Could not locate sri_lanka_flood_risk_dataset_25000.csv!')

df = pd.read_csv(dataset_path)
print(f'Successfully loaded dataset from: {dataset_path}')
print(f'Total Records (Rows):    {df.shape[0]:,}')
print(f'Total Attributes (Cols): {df.shape[1]}')

In [ ]:
# Inspect the first 5 records of the dataset
df.head()

### 2.2 Feature Dictionary & Attribute Schema
Below is the complete systematic classification of all 32 columns present in the raw dataset:

In [ ]:
# Construct a structured schema audit dataframe
schema_audit = pd.DataFrame({
    'Column Name': df.columns,
    'Data Type': df.dtypes.values,
    'Non-Null Count': df.notnull().sum().values,
    'Null Count': df.isnull().sum().values,
    'Null Percentage (%)': ((df.isnull().sum() / len(df)) * 100).round(2).values,
    'Sample Value': [df[col].iloc[0] for col in df.columns]
})

schema_audit

### 2.3 Data Integrity & Quality Audit
Let us verify missing values, duplicate observations, and check the target class balance.

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f'Total Duplicate Rows: {duplicates}')

# Verify missing value columns
missing_cols = schema_audit[schema_audit['Null Count'] > 0]
print(f'Columns with Missing Values: {len(missing_cols)}')
missing_cols[['Column Name', 'Null Count', 'Null Percentage (%)', 'Sample Value']]

In [ ]:
# Target Variable Analysis: flood_occurrence_current_event
target_counts = df['flood_occurrence_current_event'].value_counts(dropna=False)
target_pct = df['flood_occurrence_current_event'].value_counts(normalize=True, dropna=False) * 100

target_summary = pd.DataFrame({
    'Occurrences': target_counts,
    'Percentage (%)': target_pct.round(2)
})

print('=== TARGET VARIABLE DISTRIBUTION ===')
print(target_summary)

# Visual inspection of the class imbalance
plt.figure(figsize=(7, 4))
colors = ['#2b5c8f', '#d9534f']
ax = sns.barplot(x=target_counts.index, y=target_counts.values, palette=colors)
plt.title('Target Distribution: flood_occurrence_current_event', fontsize=13, fontweight='bold')
plt.xlabel('Flood Occurrence (Active Event)')
plt.ylabel('Number of Records')
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{int(height):,} ({height/len(df)*100:.1f}%)',
                (p.get_x() + p.get_width() / 2., height / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics for key physical, hydrologic, and satellite indicators
key_features = [
    'elevation_m', 'distance_to_river_m', 'rainfall_7d_mm', 
    'monthly_rainfall_mm', 'drainage_index', 'ndvi', 'ndwi', 
    'population_density_per_km2', 'flood_risk_score'
]
df[key_features].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(2)

### 2.4 Key Observations from Step 1 & Step 2
1. **Imbalance Rate (~9.89% Positive Class):** Only 2,472 out of 25,000 locations experienced active flooding during current monsoon conditions. This confirms that all subsequent modeling and cross-validation must use **stratified sampling** and **cost-sensitive learning**.
2. **Data Quality & Missing Values:**
   - `electricity` has 789 missing entries (3.16%), which will be imputed during the data cleaning stage.
   - `reason_not_good_to_live` has 18,172 missing values (72.69%), which represents post-event habitability commentary and must be dropped to prevent data leakage.
3. **Zero Duplicates:** The dataset contains 0 duplicate rows, confirming clean record indexing.
4. **Next Stages:** We are now prepared to proceed to **Step 3 (Data Understanding & Geospatial EDA)** and **Step 4 (Data Cleaning & Preprocessing)**.

## Step 3: Data Understanding & Exploratory Data Analysis (EDA)

In this section, we conduct a comprehensive exploratory data analysis to uncover hidden relationships between terrain topography, monsoon precipitation, satellite vegetation/moisture indices, and historical flood vulnerability across Sri Lanka.

In [ ]:
# 3.1 District-Level Flood Incidence Analysis
district_flood = pd.crosstab(
    df['district'], 
    df['flood_occurrence_current_event'], 
    normalize='index'
) * 100

district_flood = district_flood.sort_values(by='Yes', ascending=False)

# Visualizing district flood incidence rates
plt.figure(figsize=(14, 6))
ax = sns.barplot(x=district_flood.index, y=district_flood['Yes'], palette='Reds_r')
plt.title('Percentage of Active Flood Occurrences by District in Sri Lanka', fontsize=14, fontweight='bold')
plt.xlabel('Administrative District', fontsize=11)
plt.ylabel('Flood Incidence Rate (%)', fontsize=11)
plt.xticks(rotation=45, ha='right')

# Annotate bar percentages
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", 
                (p.get_x() + p.get_width() / 2., p.get_height() + 0.5),
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 3.2 Correlation Heatmap of Numerical Topographic & Hydrologic Features
corr_cols = [
    'elevation_m', 'distance_to_river_m', 'rainfall_7d_mm', 'monthly_rainfall_mm',
    'drainage_index', 'ndvi', 'ndwi', 'population_density_per_km2', 
    'built_up_percent', 'infrastructure_score', 'flood_risk_score'
]

# Compute Pearson correlation matrix
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(
    corr_matrix, 
    annot=True, 
    fmt='.2f', 
    cmap='coolwarm', 
    center=0, 
    linewidths=0.5, 
    cbar_kws={'label': 'Pearson Correlation'}
)
plt.title('Correlation Heatmap: Topographic, Satellite & Hydrometeorological Attributes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Satellite Earth Observation Analysis: NDWI vs. NDVI
# NDWI (Normalized Difference Water Index) vs NDVI (Normalized Difference Vegetation Index)
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df.sample(n=3000, random_state=42), # Sample for clean visual clarity
    x='ndvi', 
    y='ndwi', 
    hue='flood_occurrence_current_event', 
    palette={'No': '#2b5c8f', 'Yes': '#d9534f'},
    alpha=0.6,
    edgecolor=None
)
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)
plt.title('Satellite Spectral Analysis: NDWI (Water Saturation) vs. NDVI (Vegetation)', fontsize=13, fontweight='bold')
plt.xlabel('NDVI (Normalized Difference Vegetation Index)', fontsize=11)
plt.ylabel('NDWI (Normalized Difference Water Index)', fontsize=11)
plt.legend(title='Flood Occurrence', loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# 3.4 Elevation & River Proximity vs. Flood Occurrence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=df, 
    x='flood_occurrence_current_event', 
    y='elevation_m', 
    palette=['#2b5c8f', '#d9534f'],
    ax=axes[0]
)
axes[0].set_title('Elevation (m) Distribution by Flood Occurrence', fontweight='bold')
axes[0].set_yscale('log')
axes[0].set_ylabel('Elevation (m) [Log Scale]')

sns.boxplot(
    data=df, 
    x='flood_occurrence_current_event', 
    y='distance_to_river_m', 
    palette=['#2b5c8f', '#d9534f'],
    ax=axes[1]
)
axes[1].set_title('Distance to River (m) by Flood Occurrence', fontweight='bold')
axes[1].set_ylabel('Distance to Nearest River (m)')

plt.tight_layout()
plt.show()

### 3.5 Key Insights from EDA
1. **High-Risk Districts:** High-precipitation wet-zone districts (Ratnapura, Kalutara, Galle, Matara, Gampaha) show significantly higher historical and active flood occurrence rates compared to elevated central highlands.
2. **Topographic Constraint:** Lower elevation (<50m) and closer proximity to river channels (<1000m) exhibit a steep non-linear association with flood occurrences.
3. **Satellite Signal:** Flooded zones cluster distinctly in the upper-left quadrant of the NDWI vs NDVI space (high moisture saturation with reduced active canopy index), proving that the difference $(NDWI - NDVI)$ will be a potent engineered feature.

## Step 4: Data Cleaning & Quality Preprocessing


In this stage, we address missing values, eliminate non-predictive metadata and target leakage columns, and filter anomalies to produce a clean, production-ready dataset.

In [ ]:
# 4.1 Dropping Non-Predictive Metadata & Post-Event Leakage Columns
print(f"Initial dataset shape: {df.shape}")

# Define columns to drop:
# - Non-predictive IDs & metadata: record_id, place_name, is_synthetic, generation_date
# - Post-event target leakage: inundation_area_sqm, is_good_to_live, reason_not_good_to_live
# - Target proxy: flood_risk_score (derived directly from flood event probability)
columns_to_drop = [
    'record_id', 'place_name', 'is_synthetic', 'generation_date',
    'inundation_area_sqm', 'is_good_to_live', 'reason_not_good_to_live',
    'flood_risk_score'
]

df_cleaned = df.drop(columns=columns_to_drop)
print(f"Cleaned dataset shape after column removal: {df_cleaned.shape}")
print(f"Removed columns: {columns_to_drop}")

In [ ]:
# 4.2 Missing Value Imputation
# Audit remaining missing values:
remaining_nulls = df_cleaned.isnull().sum()
print("Remaining missing values:")
print(remaining_nulls[remaining_nulls > 0])

# Impute missing values in 'electricity' (categorical mode or 'Unknown')
electricity_mode = df_cleaned['electricity'].mode()[0]
print(f"Imputing 'electricity' null values ({df_cleaned['electricity'].isnull().sum()} records) with mode: '{electricity_mode}'")
df_cleaned['electricity'] = df_cleaned['electricity'].fillna(electricity_mode)

# Verify zero missing values remain
assert df_cleaned.isnull().sum().sum() == 0, "Data cleaning failed: Missing values still remain!"
print("✓ Zero missing values across all remaining features verified!")

In [ ]:
# 4.3 Outlier Treatment & Sanity Checks
# Inspect distributions of extreme meteorological and demographic features
outlier_candidates = ['rainfall_7d_mm', 'monthly_rainfall_mm', 'population_density_per_km2']

# Check 99th percentiles vs max to ensure physical plausibility
print("Physical Plausibility & 99th Percentile Audit:")
for col in outlier_candidates:
    p99 = df_cleaned[col].quantile(0.99)
    max_val = df_cleaned[col].max()
    min_val = df_cleaned[col].min()
    print(f"- {col:30}: Min={min_val:.1f} | 99th%={p99:.1f} | Max={max_val:.1f}")

# Verify valid physical ranges
assert (df_cleaned['elevation_m'] >= 0).all(), "Invalid negative elevation detected!"
assert (df_cleaned['distance_to_river_m'] >= 0).all(), "Invalid negative river distance detected!"
assert (df_cleaned['rainfall_7d_mm'] >= 0).all(), "Invalid negative rainfall detected!"
print("✓ Physical domain range constraints passed successfully!")

In [ ]:
# 4.4 Target Variable Binary Encoding
# Encode flood_occurrence_current_event: 'Yes' -> 1, 'No' -> 0
df_cleaned['target'] = df_cleaned['flood_occurrence_current_event'].map({'Yes': 1, 'No': 0})
df_cleaned = df_cleaned.drop(columns=['flood_occurrence_current_event'])

print("Cleaned Data Overview:")
print(f"- Feature Matrix Shape: {df_cleaned.drop(columns=['target']).shape}")
print(f"- Target Vector Shape:  {df_cleaned['target'].shape}")
print(f"- Target Class Distribution (0=No, 1=Yes):")
print(df_cleaned['target'].value_counts())

### 4.5 Summary of Data Cleaning
1. **Leakage Prevention:** Removed 8 non-predictive metadata and post-event columns (`record_id`, `place_name`, `is_synthetic`, `generation_date`, `inundation_area_sqm`, `is_good_to_live`, `reason_not_good_to_live`, `flood_risk_score`).
2. **Missing Values Resolved:** Successfully imputed all 789 missing values in `electricity` with the empirical mode (`Grid`), achieving 100% data completeness.
3. **Domain Constraints Validated:** Verified that elevation, river proximity, and rainfall values satisfy physical boundary conditions without corrupt anomalies.
4. **Ready for Next Stage:** The dataset is now clean, uncorrupted, and prepared for **Step 5 (Feature Engineering)**.